# L4: Voice as a Tool

<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> 💻 &nbsp; <b>Access <code>requirements.txt</code> and <code>helper.py</code> files:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Open"</em>.

<p> ⬇ &nbsp; <b>Download Notebooks:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Download"</em> and select <em>"Notebook (.ipynb)"</em>.</p>
</div>

<br>

<p style="background-color:#f7fff8; padding:15px; border-width:3px; border-color:#e0f0e0; border-style:solid; border-radius:6px"> 🚨
&nbsp; <b>Different run results:</b> The output generated by AI models can vary with each execution due to their dynamic, probabilistic nature. Don't be surprised if your results differ from those shown in the video.</p>

**When Your LLM Picks Up the Phone**

Voice becomes a function call your LLM chooses to invoke — same
shape as `send_email` or `query_database`. You hand Claude a
`make_phone_call` tool and watch it dial.


## Setup

You need a Vocal Bridge key, an Anthropic key, and the `vb` CLI
authenticated as the same account that owns the caller agent.

In this DeepLearning.AI environment, all three are already in
place. Outside of here, on your own machine, you'd set them
yourself.


In [ ]:
import json
import os
from pathlib import Path

from helpers import (
    load_env, vb, append_to_env, run_tool_loop,
    set_caller_purpose, place_call,
    stream_call_transcript,
    latest_session, download_recording,
)

load_env()
assert os.getenv("VOCAL_BRIDGE_API_KEY"), "Set VOCAL_BRIDGE_API_KEY in .env"
assert os.getenv("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY in .env"
print("Keys loaded ✓")


### What's in `helpers.py`

| Helper | What it does |
| --- | --- |
| `vb(*args)` | Runs a `vb` CLI command |
| `run_tool_loop()` | Drives Claude's tool-use turn end-to-end |
| `set_caller_purpose(p)` | Bakes the per-call purpose into the caller agent's prompt |
| `place_call(p)` | Sets purpose + dials the course callee, returns a sanitized result |
| `stream_call_transcript()` | Streams `vb debug` during the call |
| `latest_session()` | Gets the most recent session log |
| `download_recording(sid)` | Pulls the call audio for playback |


## Step 1 — The caller agent

The caller agent is the one *you* control. The pattern that makes
the per-call purpose actually shape the conversation:

- The caller has a **base prompt** with no specific reason for
  calling — just identity, conversational style, safety guardrails.
- The reason (the *purpose*) gets injected per call by
  `set_caller_purpose()` just before each `vb call`.

Same caller agent, different runtime context for every call.


In [ ]:
print(Path("agents/tool-caller/prompt_base.md").read_text()[:700]
      + " …")


## Step 2 — Create (or use) the caller agent

Idempotent: if `VOCAL_BRIDGE_AGENT_ID_L4` is already set (e.g. in a
sandbox), reuse it. Otherwise create a new caller agent from the
base prompt, enable outbound, and accept the outbound ToS.

In [ ]:
agent_id = os.environ.get("VOCAL_BRIDGE_AGENT_ID_L4", "").strip()
if agent_id:
    print(f"Using pre-provisioned caller: {agent_id}")
else:
    result = vb(
        "agent", "create",
        "--name", "lesson-4-tool-caller",
        "--style", "Chatty",
        "--prompt-file", "agents/tool-caller/prompt_base.md",
        "--deploy-targets", "web",
        "--debug-mode", "true",
        json_output=True,
    )
    agent_id = result["agent"]["id"]
    append_to_env("VOCAL_BRIDGE_AGENT_ID_L4", agent_id)
    vb("agent", "use", agent_id)
    vb("config", "set",
       "--outbound-enabled", "true",
       "--accept-outbound-tos")
    vb("config", "set",
       "--outbound-greeting",
       "Hi — this is an AI voice agent calling about a quick "
       "thing. Got a sec?")
    print(f"Created caller: {agent_id}")

In [ ]:
# Pin the caller as the selected agent for the per-call
# `vb prompt set` + `vb call` operations that follow.
vb("agent", "use", agent_id)
print(f"Pinned caller agent: {agent_id}")


## Step 3 — The tool schema for Claude

Standard Anthropic tool format. Note that `purpose` is **required**
— that's the bit that shapes the call.


In [ ]:
TOOLS = [
    {
        "name": "make_phone_call",
        "description": (
            "Place an outbound demo phone call to the course "
            "callee agent. Pass the purpose so the callee "
            "knows what scenario to play."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "purpose": {
                    "type": "string",
                    "description": (
                        "What the caller is calling about. "
                        "Drives the entire conversation."
                    ),
                },
                "name": {
                    "type": "string",
                    "description": "Optional friendly callee name",
                },
            },
            "required": ["purpose"],
        },
    }
]
print(json.dumps(TOOLS, indent=2))


## Step 4 — Implement the tool

`place_call()` from `helpers.py` does the work:

1. Bakes the purpose into the caller agent's prompt
   (`set_caller_purpose`).
2. Dials the course-managed callee — number lives in an env var,
   not in this notebook.
3. Returns a sanitized result (`call_id`, `status`) so Claude
   doesn't see any transport-level fields.


In [ ]:
def make_phone_call(purpose: str, name: str | None = None):
    result = place_call(purpose, name=name)
    print(f"  → call placed, purpose: {purpose}")
    return result

print("Tool ready: make_phone_call(purpose, name=None)")


## Step 5 — Let Claude decide to dial

Change the purpose below to anything you like. The callee will
play the corresponding counterpart.

Examples to try:
- *"Confirm an appointment for tomorrow at 2pm."*
- *"Check candidate availability for a Thursday interview."*
- *"Follow up on a delayed shipment, offer alternatives."*
- *"Just say hi and chat about voice AI for a minute."*

The `stream_call_transcript` helper function uses `vb debug` which streams every agent / callee / tool event from the
selected agent as the call is happening. The CLI already formats
the output — we just pass it through. Streams for up to 120
seconds, or until the call ends.


In [ ]:
USER_MESSAGE = (
    "Please place a demo call. Purpose: confirm an appointment "
    "for tomorrow at 2pm."
)

steps = run_tool_loop(
    user_message=USER_MESSAGE,
    tools=TOOLS,
    tool_handlers={"make_phone_call": make_phone_call},
    api_key=os.getenv("ANTHROPIC_API_KEY"),
)
for s in steps:
    print(s)

print("--------------------------------------------------------------------------")

stream_call_transcript(timeout_s=120)

## Step 7 — Download + play the recording

Every call is recorded server-side. Grab the most recent completed
session, pull the recording file, render it inline.


In [ ]:
session = latest_session(status="completed")
session_id = session.get("session_id") or session.get("id")
print(f"Latest session: {session_id}")
print(f"Duration: {session.get('duration_seconds', '?')}s")


In [ ]:
recording_path = download_recording(
    session_id,
    out_path=f"recording_{session_id}.mp3",
)
print(f"Downloaded → {recording_path}")


In [ ]:
from IPython.display import Audio
Audio(str(recording_path))


## What just happened

You taught an LLM to pick up the phone. The flow:

1. Claude saw the user's request, decided `make_phone_call` was
   the right tool, picked a `purpose`.
2. `place_call()` baked that purpose into the caller agent's
   prompt and shelled out to `vb call`.
3. VB dialed the course-managed callee agent. Two AI voice agents
   had a real-time conversation over real PSTN.
4. You streamed the live transcript, then downloaded the recording
   and played it back inline.

Voice is now another function call in Claude's toolbox — same
shape as `send_email` or `query_database`. Just one that rings a
phone and has a conversation.

## What's next

**Lesson 5** is the discipline layer. You'll learn how to score
calls like this one against an objective so you can ship voice
agents the way you ship code — with regression tests and a clear
quality signal.
